# 04.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/01-train-validation-test/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%


train 2,487 / test 830


## Exercise 1

$SE = \sqrt{0.7 \times 0.3 / 300} = \sqrt{0.21/300} = \sqrt{0.0007} \approx 0.0265$.

95% range ≈ 0.70 ± 2 × 0.0265 = **64.7% to 75.3%**. A 300-customer test set cannot tell 70% from 73%.

In [2]:
p, n = 0.70, 300
se = np.sqrt(p * (1 - p) / n)
print(f"SE = {se:.4f}, 95% range {p - 2*se:.1%} to {p + 2*se:.1%}")

SE = 0.0265, 95% range 64.7% to 75.3%


## Exercise 2

In [3]:
from sklearn.tree import DecisionTreeClassifier
for n in [50, 200, 800]:
    scores = []
    for s in range(30):
        Xa, Xb, ya, yb = train_test_split(cust[features], cust["churned"], test_size=n, random_state=s)
        scores.append(DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xa, ya).score(Xb, yb))
    scores = np.array(scores)
    se = np.sqrt(scores.mean() * (1 - scores.mean()) / n)
    print(f"n={n:>3}: min {scores.min():.1%}  max {scores.max():.1%}  std {scores.std():.3f}  formula SE {se:.3f}")

n= 50: min 48.0%  max 86.0%  std 0.073  formula SE 0.066
n=200: min 60.5%  max 71.5%  std 0.030  formula SE 0.033


n=800: min 63.4%  max 69.0%  std 0.014  formula SE 0.017


The observed spread shrinks roughly as $1/\sqrt{n}$, as the formula predicts: quadruple the test set, halve the wobble. (The observed spread also includes the small changes in the *training* set between splits, so it is not an exact match.)

## Exercise 3

In [4]:
from sklearn.neighbors import KNeighborsClassifier
X_tmp, X_te, y_tmp, y_te = train_test_split(cust[features], cust["churned"], test_size=0.2, random_state=42, stratify=cust["churned"])
X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

val = {k: KNeighborsClassifier(k).fit(X_tr, y_tr).score(X_val, y_val) for k in [1, 5, 15, 31, 51, 101]}
print({k: round(v, 3) for k, v in val.items()})
best_k = max(val, key=val.get)
final = KNeighborsClassifier(best_k).fit(X_tmp, y_tmp)
print(f"best k = {best_k}; test accuracy (once) = {final.score(X_te, y_te):.1%}")

{1: 0.608, 5: 0.619, 15: 0.684, 31: 0.685, 51: 0.688, 101: 0.699}
best k = 101; test accuracy (once) = 67.6%


(These features are unscaled; lesson 04.5 explains why that matters for KNN.)

## Exercise 4

In [5]:
from sklearn.tree import DecisionTreeClassifier
sept = cust
for cut in ["2011-03-01", "2011-06-01"]:
    snap = customer_table(cut)
    m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(snap[features], snap["churned"])
    print(f"trained on {cut} ({len(snap):,} customers, churn {snap['churned'].mean():.1%}) "
          f"→ September accuracy {m.score(sept[features], sept['churned']):.1%}")

trained on 2011-03-01 (1,682 customers, churn 43.1%) → September accuracy 65.5%
trained on 2011-06-01 (2,718 customers, churn 50.4%) → September accuracy 66.8%


Compare the two lines. A snapshot taken further in the past has fewer customers and a different churn rate, so its thresholds fit September less well. In production this is why models are **retrained on a schedule** (Module 10): the newest snapshot that has a complete target window is usually the best training set.